# 🖥️ Kohya Trainer XL GUI

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AlamandaCathartica/EasyTrainer/blob/main/GUI_Trainer.ipynb)

A web GUI for the **XL Lora trainer** (same pipeline as `Lora_Trainer_XL.ipynb`), with live **loss chart**, **GPU / CPU / RAM monitor**, **log viewer** and an output file list. It uses only the Python standard library.

**No setup, no uploads:** pick a **GPU runtime**, run the cell below and open the link it prints. The GUI code is downloaded automatically from GitHub into the Colab session each time. Training is started from the GUI; no other cell is needed. Google Drive is only mounted so your datasets and trained LoRAs are stored there.

The GUI is exposed through a free **Cloudflare quick tunnel** (no account needed). The link contains a secret token; anyone with it can control your training, so don't share it. Set `USE_TUNNEL = False` to use Colab's built-in port proxy instead.

In [ ]:
#@title ▶️ Start the GUI (Cloudflare tunnel)
import os, subprocess, sys, time
from google.colab import drive

if not os.path.exists('/content/drive/MyDrive'):
  drive.mount('/content/drive')

REPO_URL = 'https://github.com/AlamandaCathartica/EasyTrainer' #@param {type:"string"}
BRANCH = 'main' #@param {type:"string"}
REPO_DIR = '/content/EasyTrainer'                      # code is downloaded here every session
CACHE_DIR = '/content/drive/MyDrive/Loras/_gui'        # optional cache for cloudflared (created automatically)
GUI = os.path.join(REPO_DIR, 'gui', 'kohya_gui.py')
PORT = 7860
DEMO = False #@param {type:"boolean"}
USE_TUNNEL = True #@param {type:"boolean"}
URL_FILE = '/content/kohya_gui_url.txt'

# ---- download / update the GUI from GitHub (nothing to upload by hand) ------
subprocess.run(['pkill', '-f', 'kohya_gui.py'], check=False)
subprocess.run(['pkill', '-f', 'cloudflared'], check=False)
if os.path.isdir(os.path.join(REPO_DIR, '.git')):
  r = subprocess.run(f'git -C {REPO_DIR} fetch --depth 1 origin {BRANCH} && git -C {REPO_DIR} reset --hard FETCH_HEAD',
                     shell=True, capture_output=True, text=True)
else:
  subprocess.run(['rm', '-rf', REPO_DIR], check=False)
  r = subprocess.run(['git', 'clone', '--depth', '1', '-b', BRANCH, REPO_URL, REPO_DIR], capture_output=True, text=True)
if r.returncode != 0:
  print(r.stdout, r.stderr)
  if not os.path.exists(GUI):
    raise SystemExit(f'Could not download {REPO_URL} (branch {BRANCH}). Check the URL/branch and that the repo is public.')
  print('Update failed; using the copy already in this session.')
assert os.path.exists(GUI), f'{GUI} not found in the downloaded repo.'
print('GUI code ready:', subprocess.run(['git', '-C', REPO_DIR, 'log', '-1', '--format=%h %s'], capture_output=True, text=True).stdout.strip())

# ---- start the GUI -----------------------------------------------------------
if os.path.exists(URL_FILE): os.remove(URL_FILE)
time.sleep(1)

cmd = [sys.executable, GUI, '--port', str(PORT), '--bundle-root', CACHE_DIR, '--no-browser']
if USE_TUNNEL: cmd += ['--tunnel', '--url-file', URL_FILE]
if DEMO: cmd.append('--demo')
log = open('/content/kohya_gui.log', 'w')
proc = subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, start_new_session=True)

if USE_TUNNEL:
  print('Starting Cloudflare tunnel (first run downloads cloudflared and caches it in your Drive)...')
  for _ in range(90):
    if os.path.exists(URL_FILE) or proc.poll() is not None: break
    time.sleep(1)
  if os.path.exists(URL_FILE):
    print('\n✅ Open the GUI here (keep this link private, it contains an access token):\n')
    LINK = open(URL_FILE).read()
    print(LINK)
  else:
    print(open('/content/kohya_gui.log').read()); raise SystemExit('Tunnel did not start. See the log above.')
else:
  from google.colab import output
  time.sleep(2)
  output.serve_kernel_port_as_window(PORT)
  output.serve_kernel_port_as_iframe(PORT, height=1000)

# ---- keep this cell running ------------------------------------------------
# Colab runs one cell at a time and may reclaim a runtime that looks idle. The GUI
# starts and monitors training by itself (no other cell is needed), so this cell just
# stays alive, showing the link and a heartbeat, for as long as the GUI is up.
import json, urllib.request
from IPython.display import clear_output
def _status():
  try:
    d = json.load(urllib.request.urlopen(f'http://127.0.0.1:{PORT}/api/poll?run=-2', timeout=5))['status']
    p = d['progress']; s = f"{d['status']}"
    if p.get('total'): s += f" | step {p['step']}/{p['total']}"
    if p.get('loss') is not None: s += f" | loss {p['loss']:.4f}"
    return s
  except Exception as e:
    return f'GUI not responding ({e})'
try:
  while proc.poll() is None:
    clear_output(wait=True)
    if USE_TUNNEL: print('GUI link (keep private):\n' + LINK)
    print('\n' + time.strftime('%H:%M:%S'), '-', _status())
    print('\nLeave this cell running. Stopping it does not stop the GUI or training; use Stop in the GUI.')
    time.sleep(20)
  print('The GUI server exited. See /content/kohya_gui.log')
except KeyboardInterrupt:
  print('Cell stopped. The GUI keeps running in the background.')


Closing the browser tab doesn't stop training; use **Stop** in the GUI. Server log: `/content/kohya_gui.log`.